# Part 1: Sequence Probabilities & Next-Token Distributions

A **hidden Markov model** (HMM) generates a stream of observable symbols while transitioning among a set of hidden states you never see directly. We package the whole process into a single **observation-indexed transition tensor** $T$ of shape `(n_obs, n_states, n_states)`:

$$
T[x, i, j] = P(\text{emit symbol } x,\ \text{state } j \; | \; \text{state } i),
$$

and an **initial distribution vector** $\eta^{(\emptyset)}$ of shape `(n_states,)`:

$$
\eta^{(\emptyset)}[i] = P(\text{state } i).
$$

Probabilities are assigned to sequences $x_{1:n} = (x_1, x_2, \ldots, x_n)$ via the expression

$$
P(x_{1:n}) = \eta^{(\emptyset)}T[x_1]T[x_2]\cdots T[x_n] \mathbf{1}
$$

where $\mathbf{1}$ is the all-ones column vector i.e., $\mathbf{1}_i = 1$ for all $i$. 

From sequence probabilities everything else follows. The next-token distribution after a prefix $x_{1:n}$ is the ratio of joints,

$$
P(x_{n+1} = k \mid x_{1:n}) = \frac{P(x_{1:n}, k)}{P(x_{1:n})} = \frac{\eta^{(\emptyset)}T[x_1]T[x_2]\cdots T[x_n]T[k] \mathbf{1}}{\eta^{(\emptyset)}T[x_1]T[x_2]\cdots T[x_n] \mathbf{1}},
$$

which is well-defined exactly when the prefix has nonzero probability.

## What you'll implement

In this part you build the forward machinery from the bottom up, in order:

1. `validate` — assert the process is well-formed (stochastic transition matrix and initial vector).
2. `_propagate` — multiply the `initial_vect` by a sequence of transition matrices.
3. `sequence_probability` — the probability of sampling a sequence from the HMM.
4. `conditional_next_token_probabilities` — the next-token distribution over symbols `(n_obs,)`, conditional on a sequence prefix.
5. `all_next_token_probabilities` — every reachable prefix up to a given depth mapped to its next-token distribution, built breadth-first with zero-probability branches pruned.

## The monkeypatch workflow

You're given an `HMM` skeleton with just an `__init__` (it stores `transition_tensor` and `initial_vect`). Each exercise asks you to define a standalone function, attach it to the class, then run the matching test:

```python
def _propagate(self, sequence, vect=None):
    ...

HMM._propagate = _propagate
tests.test_propagate(HMM)
```

Defining methods this way lets each exercise stand on its own while accumulating onto the same class. Later methods may call ones you defined earlier, so keep them attached as you go.

## Example processes

Two processes from `processes.py` serve as running fixtures:

- **Z1R** (Zero-One-Random): 3 hidden states, **2 symbols**. Emits `0` then `1` deterministically, then a random bit. A clean sanity check with sharp, mostly-deterministic structure.
- **Mess3**: 3 hidden states, **3 symbols**. A symmetric process whose belief states trace out a fractal in the 2-simplex — a richer testbed for the distributions you compute here and the belief-state geometry explored later.

## Setup

In [ ]:
# Best-effort auto-reload of edited modules (silently skipped where IPython's
# autoreload extension is unavailable, e.g. some Colab Python 3.12 images).
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import sys
import pathlib

# Make the exercise modules (processes / solutions / tests / plotting) importable
# regardless of the kernel's working directory (the exercises folder or the repo root).
_here = pathlib.Path.cwd()
for _cand in [_here, _here / "exercises"]:
    if (_cand / "tests.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
from collections.abc import Sequence

import processes
import solutions
import tests

# Shape aliases (documentation only -- pyright sees plain np.ndarray).
type TransitionTensor = np.ndarray  # (n_obs, n_states, n_states)
type StateVector = np.ndarray       # (n_states,) -- belief / propagated state vector
type TokenDist = np.ndarray         # (n_obs,)    -- distribution over next tokens

---

## Exercises

The cells above are **setup**. First run the cell below to define the `HMM` class, then work through the exercises in order: for each, complete the `# YOUR CODE HERE` cell (it attaches the method to `HMM` via `HMM.<name> = <name>`), then run its `tests.test_<name>(HMM)` cell — it prints **All tests passed!** when your implementation is correct.

The `HMM` skeleton — each exercise adds a method to this class via `HMM.<name> = <name>`.

In [ ]:
class HMM:
    """HMM defined by an observation-indexed transition tensor and an initial vector.

    Shapes
    ------
    transition_tensor        : (n_obs, n_states, n_states)
    belief / state vectors   : (n_states,)
    next-token distributions : (n_obs,)
    """

    def __init__(self, transition_tensor: TransitionTensor, initial_vect: StateVector) -> None:
        self.transition_tensor = np.asarray(transition_tensor)
        self.initial_vect = np.atleast_1d(np.asarray(initial_vect)).ravel()

### Exercise - implement `validate`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~10 minutes on this exercise.

Before computing anything, it is worth checking that `transition_tensor` and `initial_vect` actually define a valid hidden Markov model. Implement `validate`, which `assert`s the structural constraints (and so raises `AssertionError` on a malformed process).

Two conditions must hold:

- **The transition matrix is stochastic.** Every entry of $T$ is a probability, so $T \ge 0$. Summing over the symbol axis collapses the emissions into the underlying state-transition matrix $T_{ij} = \sum_x T[x, i, j] = P(j \;|\; i)$, which must be **row-stochastic**: each row sums to $1$ (from any state, *some* (symbol, next-state) pair occurs with total probability $1$).

- **The initial vector is stochastic.** $\text{initial\_vect} \ge 0$ and $\sum_i \text{initial\_vect}_i = 1$.

Write one `assert` per condition with an informative message; use `np.allclose` / `np.isclose` so floating-point round-off does not trip the checks.

<details><summary>Hint</summary>

`self.transition_tensor.sum(axis=0)` is the state-transition matrix $T$; its row sums are `T.sum(axis=1)`. The four checks are `(self.transition_tensor >= 0).all()`, `np.allclose(T.sum(axis=1), 1.0)`, `(self.initial_vect >= 0).all()`, and `np.isclose(self.initial_vect.sum(), 1.0)`.

</details>

In [ ]:
def validate(self) -> None:
    """Assert that transition_tensor and initial_vect define a valid HMM.

    The observation-summed state-transition matrix must be row-stochastic and
    the initial vector must be a probability distribution. Raises AssertionError
    otherwise.
    """
    assert (self.transition_tensor >= 0).all(), "transition_tensor must be non-negative"
    transition_matrix = self.transition_tensor.sum(axis=0)
    assert np.allclose(transition_matrix.sum(axis=1), 1.0), \
        "summed transition matrix must be row-stochastic (each row sums to 1)"
    assert (self.initial_vect >= 0).all(), "initial_vect must be non-negative"
    assert np.isclose(self.initial_vect.sum(), 1.0), "initial_vect must sum to 1"

HMM.validate = validate

In [ ]:
tests.test_validate(HMM)

### Exercise - implement `_propagate`

> **Difficulty:** 🔴🔴⚪⚪⚪  
> **Importance:** 🔵🔵🔵🔵⚪  
> 
> You should spend up to ~10-15 minutes on this exercise.

This is the workhorse that nearly every other method calls. Given an observation `sequence` $x_1, x_2, \dots, x_L$, it runs the **forward recursion** over hidden states, starting from a row vector (`initial_vect` by default, or a supplied `vect`) and right-multiplying by the observation-conditioned matrix at each step.

Concretely, with `transition_tensor` $T$ of shape `(n_obs, n_states, n_states)` where $T[x, i, j] = P(\text{emit } x,\ j\;|\; i)$, the recursion is

$$
\alpha^{(\emptyset)} = \texttt{vect}, \qquad \alpha^{(x_{1:t})} = \alpha^{(x_{1:t-1})}\, T[x_t], \qquad \text{return } \alpha^{(x_{1:L})}.
$$

The result is the **unnormalized** forward state vector $\alpha^{(x_{1:L})}$ of shape `(n_states,)`. Component $j$ accumulates the total probability mass of all hidden-state paths consistent with the observed prefix that end in state $j$; hence $\sum_j \alpha^{(x_{1:L})}_j = P(x_1, \dots, x_L)$. Do not normalize here — downstream callers rely on the raw mass.

<details><summary>Hint</summary>

Default `vect` to `self.initial_vect` when `None`. Then loop over each `obs` in the sequence, updating `vect` with the row-vector-times-matrix product:

```python
vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
```

Return `vect` after the loop (it is $\alpha^{(x_{1:L})}$).

</details>

In [ ]:
def _propagate(self, sequence: Sequence[int], vect: StateVector | None = None) -> StateVector:
    """Run a sequence through the tensor, returning the unnormalized state vector."""
    vect = self.initial_vect if vect is None else vect
    for obs in sequence:
        vect = np.einsum("i,ij->j", vect, self.transition_tensor[obs])
    return vect

HMM._propagate = _propagate

In [ ]:
tests.test_propagate(HMM)

### Exercise - implement `sequence_probability`

> **Difficulty:** 🔴⚪⚪⚪⚪  
> **Importance:** 🔵🔵🔵⚪⚪  
> 
> You should spend up to ~5-10 minutes on this exercise.

This method returns the total probability that the process emits exactly the given observation sequence, $P(x_{1:L})$. With `_propagate` already in hand, you receive the (unnormalized) forward vector $\alpha^{(x_{1:L})} \in \mathbb{R}^{n\_states}$, whose component $\alpha_j^{(x_{1:L})}$ is the joint probability of having emitted $x_{1:L}$ *and* landing in state $j$:

$$
\alpha_j^{(x_{1:L})} = P(x_{1:L},\, s_L = j).
$$

To get the marginal probability of the sequence, sum out the unobserved final state. Summing $\alpha$ over $j$ contracts the forward vector with the all-ones final vector $\mathbf{1}$, which is the correct termination step for a standard HMM (every state is an admissible stopping point):

$$
P(x_{1:L}) = \alpha^{(x_{1:L})} \mathbf{1} = \sum_j \alpha_j^{(x_{1:L})}.
$$

<details><summary>Hint</summary>

Call `_propagate` to obtain $\alpha$, then reduce it to a scalar by summing over the state axis (the contraction with $\mathbf{1}$).

</details>

In [ ]:
def sequence_probability(self, sequence: Sequence[int]) -> float:
    return np.sum(self._propagate(sequence))

HMM.sequence_probability = sequence_probability

In [ ]:
tests.test_sequence_probability(HMM)

### Exercise - implement `conditional_next_token_probabilities`

> **Difficulty:** 🔴⚪⚪⚪⚪  
> **Importance:** 🔵🔵🔵🔵⚪  
> 
> You should spend up to ~5-10 minutes on this exercise.


Given an observed sequence $x_{1:L}$, this returns the distribution over the next symbol, $P(\text{next} = k \mid x_{1:L})$ for each $k \in \{0, \dots, n_{obs}-1\}$. The result is a `TokenDist` of shape `(n_obs,)`.

Apply the definition of conditional probability directly:

$$
P(\text{next} = k \mid x_{1:L}) = \frac{P(x_{1:L},\, k)}{P(x_{1:L})}
$$

Both numerator and denominator are values you can already obtain from `sequence_probability`: the numerator is the probability of the sequence with $k$ appended, and the denominator is the probability of the sequence on its own. No belief states are needed — this is purely a ratio of sequence probabilities.

If $P(x_{1:L}) = 0$ the conditional is undefined (division by zero), so raise a `ValueError` rather than returning anything.

<details><summary>Hint</summary>

No einsum here. Compute the denominator once as `sequence_probability(sequence)`; if it is `0`, raise `ValueError`. Then, for each candidate next symbol $k$ in `range(n_obs)`, evaluate `sequence_probability(sequence + [k])` and divide by the denominator. Collect the `n_obs` ratios into a single vector.

</details>

In [ ]:
def conditional_next_token_probabilities(self, sequence: Sequence[int]) -> TokenDist:
    seq = list(sequence)
    p_seq = self.sequence_probability(seq)
    if np.allclose(p_seq, 0.0):
        raise ValueError(f"sequence {tuple(sequence)} has zero probability; "
                         "next-token distribution is undefined.")
    n_obs = self.transition_tensor.shape[0]
    joint = np.array([self.sequence_probability(seq + [k]) for k in range(n_obs)])
    return joint / p_seq

HMM.conditional_next_token_probabilities = conditional_next_token_probabilities

In [ ]:
tests.test_conditional_next_token_probabilities(HMM)

### Exercise - implement `all_next_token_probabilities`

> **Difficulty:** 🔴🔴🔴⚪⚪  
> **Importance:** 🔵🔵⚪⚪⚪  
> 
> You should spend up to ~15-20 minutes on this exercise.

This method enumerates the next-token distribution for *every reachable* observation sequence up to length `max_depth`, returning a dict keyed by the sequence tuple. The empty sequence `()` maps to the prior next-token distribution `conditional_next_token_probabilities(())`, and a sequence like `(2, 0)` maps to $P(x_3 \mid x_{1:2} = (2, 0))$. Each value is a `TokenDist` of shape `(n_obs,)`.

Proceed breadth-first. Start from the frontier $\{()\}$ with its next-token distribution. From a frontier sequence `seq` with distribution `ntp`, an observation `obs` is a *reachable* continuation iff $\text{ntp}[\text{obs}] > 0$; the conditional distribution of `seq + (obs,)` is then well defined and is obtained from `conditional_next_token_probabilities`. Continuations with $\text{ntp}[\text{obs}] = 0$ have probability zero, so the conditional state is undefined — prune those branches rather than recursing into them.

This is the exhaustive, exact counterpart to sampling: instead of following one trajectory, it expands the full reachable tree of histories, which makes it useful for computing exact statistics (e.g. entropies, myopic predictions) over all paths the process can produce.

<details><summary>Hint</summary>

No einsum here — this is plain tree traversal. Maintain a result dict and a frontier (queue) of sequences whose next-token distributions you have. For each `seq` in the current frontier, store its `ntp` in the result, then if `len(seq) < max_depth`, loop over `obs in range(n_obs)` and enqueue `seq + (obs,)` only when `ntp[obs]` is nonzero, computing its distribution via `self.conditional_next_token_probabilities(seq + (obs,))`. Seed the loop with `seq = ()`.

</details>

In [ ]:
def all_next_token_probabilities(self, max_depth: int) -> dict[tuple[int, ...], TokenDist]:
    """Next-token distribution for every reachable sequence of length 0..max_depth.

    Built breadth-first; a continuation is reachable iff its probability in the
    current distribution is nonzero (zero-probability branches are pruned).
    Uses conditional_next_token_probabilities directly -- no belief-state machinery.
    Includes () -> the prior next-token distribution.
    """
    prior = self.conditional_next_token_probabilities(())
    result = {(): prior}
    frontier = [((), prior)]
    for _ in range(max_depth):
        next_frontier = []
        for seq, ntp in frontier:
            for obs in range(len(ntp)):
                if np.allclose(ntp[obs], 0.0):
                    continue  # zero-probability continuation -> prune
                new_seq = seq + (obs,)
                child = self.conditional_next_token_probabilities(new_seq)
                result[new_seq] = child
                next_frontier.append((new_seq, child))
        frontier = next_frontier
    return result

HMM.all_next_token_probabilities = all_next_token_probabilities

In [ ]:
tests.test_all_next_token_probabilities(HMM)

## Demo

With the methods implemented, we can explore a process. The cells below render the **next-token-distribution geometry** interactively (via the local `plotting.py`): every reachable sequence's next-token distribution, plotted on the symbol simplex and colored by its coordinates. Hover a point to see the sequence that induces it. We show two 3-symbol processes — **Mess3** (3 states; a fractal) and **arch** (4 states).

In [ ]:
from itertools import product

import plotting

hmm = HMM(processes.mess3(0.15, 0.2), processes.uniform_initial(3))
hmm.validate()  # a well-formed process passes silently

# A proper process: probabilities over all length-3 sequences sum to 1.
total = sum(hmm.sequence_probability(s) for s in product(range(3), repeat=3))
print("sum P(length-3 sequences) =", round(float(total), 6))

print("P(next | (0, 1))           =", np.round(hmm.conditional_next_token_probabilities((0, 1)), 4))
print("# reachable seqs <= depth 4:", len(hmm.all_next_token_probabilities(4)))

# Interactive next-token-distribution geometry (uses the local plotting.py).
# Each point is the next-token distribution after some sequence, plotted on the
# symbol simplex and colored by its coordinates; hover shows the inducing sequence.
depth = 7
ntps = hmm.all_next_token_probabilities(depth)
plotting.plot_next_token_distributions(
    np.array([np.real(v) for v in ntps.values()]),
    sequences=list(ntps.keys()),
    title=f"Mess3 next-token-distribution geometry (depth {depth})",
)

In [ ]:
# The arch process: 4 hidden states, 3 symbols. Its next-token distributions still
# live on the 3-symbol simplex (a 2D triangle), but trace a different geometry.
arch_hmm = HMM(processes.arch(0.6), processes.uniform_initial(4))
arch_hmm.validate()

arch_ntps = arch_hmm.all_next_token_probabilities(depth)
plotting.plot_next_token_distributions(
    np.array([np.real(v) for v in arch_ntps.values()]),
    sequences=list(arch_ntps.keys()),
    title=f"Arch next-token-distribution geometry (depth {depth})",
)

### Bring your own process

The processes above come from `processes.py`, but nothing is special about them — **any** non-negative transition tensor whose observation-summed matrix is row-stochastic defines a valid HMM. Here we define one **inline** (not in `processes.py`): the **Fern** process (3 states, 2 symbols). `validate()` confirms it is well-formed, then it runs through exactly the same machinery. With only 2 symbols, its next-token distributions live on a 1-simplex (a line segment).

In [ ]:
def fern(x: float) -> np.ndarray:
    """Fern process (custom): 3 hidden states, 2 symbols."""
    assert 0.0 <= x <= 1.0
    return np.array([
        [[0.3942, 0.00512, 0.0381], [0.0, 0.53, 0.0], [0.0, 0.326 * x, 0.554]],
        [[0.3358, 0.01088, 0.2159], [0.0, 0.0, 0.47], [0.12, 0.326 * (1 - x), 0.0]],
    ])

fern_hmm = HMM(fern(0.5), processes.uniform_initial(3))
fern_hmm.validate()  # a custom process still has to be a valid HMM

fern_depth = 12
fern_ntps = fern_hmm.all_next_token_probabilities(fern_depth)
plotting.plot_next_token_distributions(
    np.array([np.real(v) for v in fern_ntps.values()]),
    sequences=list(fern_ntps.keys()),
    title=f"Fern next-token-distribution geometry (depth {fern_depth})",
)

In [ ]:
hmm = HMM(processes.mess3(0.15, 0.2), processes.uniform_initial(3))

hmm.all_next_token_probabilities(3)